# Portable LightGBM Robot-Health Submission

This notebook trains the model **from `train.csv` only** and predicts `test.csv`.

### Changes from the original Grok script
- Removed all use of `1.csv`.
- Removed hard-coded `/home/workdir/...` paths.
- Automatically searches for `train.csv` and `test.csv`.
- Works in Google Colab, Jupyter, Kaggle-style environments, or a normal Python environment.
- Saves the final submission as `submission.csv`.
- The submission contains exactly `row_id,target`.


In [ ]:
# Install dependencies if they are missing.
# In Google Colab/Jupyter, run this cell once.
%pip install -q pandas numpy scikit-learn lightgbm


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
import lightgbm as lgb

# -------------------------------------------------------------------
# Portable data-location logic
# -------------------------------------------------------------------
# Option 1: Put train.csv and test.csv in the same folder as this notebook.
# Option 2: Set DATA_DIR manually, e.g.:
# DATA_DIR = "/content/drive/MyDrive"
#
# Leave DATA_DIR = None for automatic searching.

DATA_DIR = None

def find_file(filename, data_dir=None):
    if data_dir:
        p = Path(data_dir) / filename
        if p.exists():
            return p

    candidates = [
        Path.cwd() / filename,
        Path("/content") / filename,
        Path("/content/drive/MyDrive") / filename,
        Path("/kaggle/working") / filename,
        Path("/kaggle/input") / filename,
        Path("/home/oai/share") / filename,
        Path("/mnt/data") / filename,
    ]

    # Search a few common parent locations without assuming a platform.
    for root in [Path.cwd(), Path("/content"), Path("/kaggle/input"), Path("/mnt/data")]:
        if root.exists():
            try:
                candidates.extend(root.rglob(filename))
            except (PermissionError, OSError):
                pass

    seen = set()
    for p in candidates:
        p = Path(p)
        if p not in seen and p.exists() and p.is_file():
            seen.add(p)
            return p

    raise FileNotFoundError(
        f"Could not find {filename}. Put it beside this notebook or set DATA_DIR "
        f"to the folder containing train.csv and test.csv."
    )

train_path = find_file("train.csv", DATA_DIR)
test_path = find_file("test.csv", DATA_DIR)

print("Train:", train_path)
print("Test :", test_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape :", test.shape)


In [ ]:
# Basic validation
required_train = {"row_id", "target"}
required_test = {"row_id"}

missing_train = required_train - set(train.columns)
missing_test = required_test - set(test.columns)

if missing_train:
    raise ValueError(f"train.csv is missing required columns: {sorted(missing_train)}")
if missing_test:
    raise ValueError(f"test.csv is missing required columns: {sorted(missing_test)}")

if len(test) == 0:
    raise ValueError("test.csv contains no rows.")

print("Target classes:", sorted(train["target"].dropna().unique().tolist()))
print("Test rows:", len(test))


In [ ]:
# -------------------------------------------------------------------
# Feature preparation
# -------------------------------------------------------------------
# Exclude identifiers, target, and the two columns intentionally excluded
# in the original model.
base = [
    c for c in train.columns
    if c not in ["row_id", "target", "motion_elegance_score", "shift_reliability_index"]
]

# Make sure every base feature exists in test.csv.
missing_test_features = [c for c in base if c not in test.columns]
if missing_test_features:
    raise ValueError(
        "test.csv is missing training feature columns: "
        + ", ".join(missing_test_features)
    )

X = train[base].copy()
Xt = test[base].copy()
y = train["target"].values

# Data-quality features.
for d in [X, Xt]:
    d["battery_over"] = (d["battery_health_pct"] > 100).astype(int)
    d["battery_health_pct"] = d["battery_health_pct"].clip(0, 100)
    d["joint_missing"] = d["joint_torque_variance"].isna().astype(int)

# Median imputation fitted ONLY on training data.
imp = SimpleImputer(strategy="median")
X[base] = imp.fit_transform(X[base])
Xt[base] = imp.transform(Xt[base])

# Interaction / ratio features.
for d in [X, Xt]:
    d["error_rate"] = d["error_count_7d"] / (d["shift_hours_last_7d"] + 1)
    d["health"] = (
        d["battery_health_pct"] / 100
        + d["sensor_calibration_score"]
        + d["self_diagnostic_score"] / 100
        + d["load_capacity_pct"]
        + d["task_completion_rate"]
        + d["vision_accuracy"]
    ) / 6
    d["torque_temp"] = d["joint_torque_variance"] * d["motor_temperature_c"]
    d["cycle_err"] = d["avg_cycle_time_sec"] * d["error_count_7d"]
    d["maint_up"] = d["last_maintenance_days"] * d["uptime_hrs"]
    d["vision_task"] = d["vision_accuracy"] * d["task_completion_rate"]
    d["diag_sens"] = d["self_diagnostic_score"] * d["sensor_calibration_score"]
    d["temp_load"] = d["motor_temperature_c"] * d["load_capacity_pct"]
    d["err_up"] = d["error_count_7d"] / (d["uptime_hrs"] + 0.5)

feats = list(X.columns)

print("Original features:", len(base))
print("Final features   :", len(feats))


In [ ]:
# -------------------------------------------------------------------
# Train LightGBM
# -------------------------------------------------------------------
model = lgb.LGBMClassifier(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=8,
    num_leaves=48,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model.fit(X[feats], y)

proba = model.predict_proba(Xt[feats])
pred = model.classes_[proba.argmax(axis=1)]
conf = proba.max(axis=1)

print("Prediction distribution:")
print(pd.Series(pred).value_counts().sort_index())
print("Mean confidence:", round(float(conf.mean()), 4))


In [ ]:
# -------------------------------------------------------------------
# Create final submission
# -------------------------------------------------------------------
# IMPORTANT:
# No previous submission/result is used here.
# Every test prediction comes directly from the newly trained model.

sub = pd.DataFrame({
    "row_id": test["row_id"].values,
    "target": pred
})

# Preserve the required two-column format.
sub = sub[["row_id", "target"]]

# Validate before saving.
assert len(sub) == len(test), "Submission row count does not match test.csv."
assert list(sub.columns) == ["row_id", "target"], "Incorrect submission columns."
assert sub["row_id"].is_unique, "row_id must be unique in the submission."

output_path = Path.cwd() / "submission.csv"
sub.to_csv(output_path, index=False)

print("\nFinal target distribution:")
print(sub["target"].value_counts().sort_index())
print(f"\nSaved: {output_path.resolve()}")
print(f"Rows : {len(sub)}")


In [ ]:
# Preview the final submission
display(sub.head(10))
